In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os 

os.chdir('ML_Pipeline')

from ML_train import train_and_select_best_model, evaluate_model, inverse_design_bo3, inverse_design_cma, run_nsga2, run_nsga3

from run_MD_sim_for_K import MD_sim


# === 1. Load Data ===
df_full = pd.read_csv('ML_Pipeline/data/AL_data.csv')
df_full = df_full.dropna()

X_full = df_full.values[:, :4]
y_full = df_full.values[:, 4:]

prop_dict = {}
    
for i, prop in enumerate(['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']):
    prop_dict[f'Property {i+1}'] = prop

# Define target property values and tolerable errors
target_y = np.array([2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47])
targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])  # in %

# Calculate percentage error
y_error_fulll = (100 * (np.abs(y_full - target_y) / target_y))


In [ ]:
for i, prop in enumerate(['error_D_11', 'error_D_11H', 'error_D_14', 'error_SE_T11', 'error_SE_T11H', 'error_SE_T14', 'error_BM_T11', 'error_BM_T11H', 'error_BM_T14']):
    df_full[prop] = y_error_fulll[:, i]
    

In [ ]:
df_clean  = df_full.iloc[(y_error_fulll < np.array([100, 1, 1, 100, 100, 100, 100, 100, 100])).sum(axis=1) == 9]


In [ ]:
df_clean[['sc_r', 'sc_eps', 'oc_r', 'oc_eps']]


In [ ]:
# df_clean  = df_full.iloc[(y_error_fulll < np.array([100, 1, 1, 100, 100, 100, 100, 100, 100])).sum(axis=1) == 9]
df_clean  = df_full.iloc[(y_error_fulll < np.array([100, 1, 1, 100, 100, 100, 15, 15, 100])).sum(axis=1) == 9]
# df_clean  = df_full.iloc[(y_error_fulll < np.array([100, 1, 1, 100, 10, 10, 10, 10, 100])).sum(axis=1) == 9]
# df_clean  = df_full.iloc[(y_error_fulll < np.array([100, 1, 1, 100, 5, 5, 10, 10, 100])).sum(axis=1) == 9]
df_clean

In [ ]:
import numpy as np

ocx_r = 3.70
ocx_eps = 0.12
ktheta_sets = [170, 150, 100, 80]

all_params = np.array([
    [
        float(df_clean['sc_r'].values[i]),
        float(df_clean['sc_eps'].values[i]),
        float(df_clean['oc_r'].values[i]),
        float(df_clean['oc_eps'].values[i]),
        ocx_r,
        ocx_eps,
        k,
        k
    ]
    for i in range(len(df_clean))
    for k in ktheta_sets
])
print(all_params.shape)

In [ ]:
# === 7. Run MD simulation ===

df_new_ocx_ktheta = MD_sim(all_params, optimizer= 'ocx_ktheta',)
df_new_ocx_ktheta = df_new_ocx_ktheta.dropna()

# === 8. Evaluate match ===
md_prop_pred = df_new_ocx_ktheta.values[:,4:]
y_error = np.abs(100*(md_prop_pred - np.array(target_y))/np.array(target_y))

for i in range(len(y_error)):
    targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])
    if (y_error[i] <= targeted_error).sum() == 9:
        print("All properties are within error limits")
        print(df_new_ocx_ktheta.iloc[i])
        break # break Active learning loop.

    else:
        total_false = (~(y_error[i] <= targeted_error)).sum()
        print("❌ Failure! Outside acceptable error range.")
        print(f"Failed for {total_false} properties")
        print("Error percentage:", y_error[i].round(2))
